# Apartment Room Classifier — ViT Transfer Learning

Fine-tunes **`google/vit-base-patch16-224`** on apartment-relevant room categories
from the **MIT Indoor Scenes** dataset, then pushes the trained model to the Hugging Face Hub.

**Room categories used:**
- bathroom, bedroom, children\'s room, corridor, dining room, kitchen, living room, nursery

> **Tip:** Run on Google Colab (GPU runtime) for fast training.

In [ ]:
# Install dependencies (only needed in Colab / fresh env)
!pip install transformers datasets evaluate torch torchvision pillow \
             huggingface_hub accelerate scikit-learn -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from datasets import DatasetDict, load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, Trainer, TrainingArguments
import evaluate

from dotenv import load_dotenv
load_dotenv()

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

In [ ]:
# Login to Hugging Face Hub (required for pushing model)
# Add HF_TOKEN to your .env file or enter it here
from huggingface_hub import notebook_login
notebook_login()

## 1. Load & Filter Dataset

We use the **MIT Indoor Scenes** dataset from Hugging Face and keep only the
room types that are relevant to apartments.

In [ ]:
# Load MIT Indoor Scenes (via keremberke repackaging)
raw_dataset = load_dataset("keremberke/indoor-scene-classification", name="full", trust_remote_code=True)

# Inspect all available class labels
all_labels = raw_dataset["train"].features["label"].names
print(f"Total classes: {len(all_labels)}")
print(sorted(all_labels))

In [ ]:
# Apartment-relevant room types (adjust if class names differ in the dataset)
APARTMENT_CLASSES = [
    "bathroom",
    "bedroom",
    "children_room",
    "corridor",
    "dining_room",
    "kitchen",
    "livingroom",
    "nursery",
]

# Keep only classes that actually exist in the dataset
APARTMENT_CLASSES = [c for c in APARTMENT_CLASSES if c in all_labels]
print(f"Using {len(APARTMENT_CLASSES)} classes: {APARTMENT_CLASSES}")

# Map original label ids to the subset we care about
keep_ids = {all_labels.index(c) for c in APARTMENT_CLASSES}

In [ ]:
def filter_and_remap(batch):
    """Keep only apartment-relevant samples and remap label IDs to 0-based."""
    return [l in keep_ids for l in batch["label"]]

# Filter both splits
filtered = DatasetDict()
for split in raw_dataset.keys():
    filtered[split] = raw_dataset[split].filter(
        filter_and_remap, batched=True, batch_size=256
    )

print(filtered)

In [ ]:
# Build new 0-based label mappings
label2id = {label: idx for idx, label in enumerate(APARTMENT_CLASSES)}
id2label = {idx: label for idx, label in enumerate(APARTMENT_CLASSES)}

# Human-readable display names for the app
DISPLAY_NAMES = {
    "bathroom": "bathroom",
    "bedroom": "bedroom",
    "children_room": "children's room",
    "corridor": "corridor",
    "dining_room": "dining room",
    "kitchen": "kitchen",
    "livingroom": "living room",
    "nursery": "nursery",
}

# Remap labels to new 0-based IDs
orig_label_to_new_id = {all_labels.index(c): label2id[c] for c in APARTMENT_CLASSES}

def remap_labels(example):
    example["label"] = orig_label_to_new_id[example["label"]]
    return example

filtered = filtered.map(remap_labels)
print(label2id)
print(id2label)

## 2. Train / Validation / Test Split

In [ ]:
# Use existing train split; create val + test from it if no separate val exists
if "validation" in filtered and "test" in filtered:
    dataset = filtered
elif "test" in filtered:
    # train → 80 / val → 10 / test → existing
    split = filtered["train"].train_test_split(test_size=0.1, seed=42)
    dataset = DatasetDict({
        "train": split["train"],
        "validation": split["test"],
        "test": filtered["test"],
    })
else:
    split1 = filtered["train"].train_test_split(test_size=0.2, seed=42)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=42)
    dataset = DatasetDict({
        "train": split1["train"],
        "validation": split2["train"],
        "test": split2["test"],
    })

print(dataset)

## 3. Sample Images

In [ ]:
def show_samples(ds, rows=2, cols=4):
    samples = ds.shuffle(seed=0).select(range(rows * cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for i, ax in enumerate(axes.flat):
        img = samples[i]["image"].convert("RGB")
        label = id2label[samples[i]["label"]]
        ax.imshow(img)
        ax.set_title(DISPLAY_NAMES.get(label, label), fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(dataset["train"])

## 4. Preprocessing

In [ ]:
BASE_MODEL = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
processor

In [ ]:
def transforms(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

processed = dataset.with_transform(transforms)

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["labels"] for x in batch]),
    }

## 5. Metric

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

## 6. Load & Freeze Model

In [ ]:
model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(APARTMENT_CLASSES),
    id2label={k: DISPLAY_NAMES.get(v, v) for k, v in id2label.items()},
    label2id={DISPLAY_NAMES.get(k, k): v for k, v in label2id.items()},
    ignore_mismatched_sizes=True,
)

# Freeze all layers except the classifier head
for name, p in model.named_parameters():
    if not name.startswith("classifier"):
        p.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

## 7. Training

In [ ]:
HF_REPO_ID = "Scampolonii/vit-apartment-rooms"  # Change to your HF username

training_args = TrainingArguments(
    output_dir="./vit-apartment-rooms",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    num_train_epochs=5,
    learning_rate=3e-4,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=processed["train"],
    eval_dataset=processed["validation"],
    processing_class=processor,
)

trainer.train()

## 8. Evaluate on Test Set

In [ ]:
metrics = trainer.evaluate(processed["test"])
print("Test metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## 9. Visualise Predictions

In [ ]:
def show_predictions(rows=2, cols=4):
    samples = dataset["test"].shuffle(seed=1).select(range(rows * cols))
    proc_samples = samples.with_transform(transforms)
    preds = trainer.predict(proc_samples).predictions.argmax(axis=1)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for i, ax in enumerate(axes.flat):
        true_label = DISPLAY_NAMES.get(id2label[samples[i]["label"]], id2label[samples[i]["label"]])
        pred_label = DISPLAY_NAMES.get(id2label[preds[i]], id2label[preds[i]])
        color = "green" if true_label == pred_label else "red"
        ax.imshow(samples[i]["image"].convert("RGB"))
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions()

## 10. Push Model & Processor to Hugging Face Hub

In [ ]:
# Save locally
trainer.save_model("./vit-apartment-rooms")
processor.save_pretrained("./vit-apartment-rooms")

# Push to Hub
trainer.push_to_hub(commit_message="final model: ViT fine-tuned on apartment room classification")
processor.push_to_hub(HF_REPO_ID)

print(f"Model pushed to: https://huggingface.co/{HF_REPO_ID}")

## 11. Quick Inference Test (before deploying the app)

In [ ]:
from transformers import pipeline

clf = pipeline("image-classification", model=HF_REPO_ID)

# Test with a sample from the test set
sample_img = dataset["test"][0]["image"]
true_label = DISPLAY_NAMES.get(id2label[dataset["test"][0]["label"]], "?")
results = clf(sample_img)

print(f"True label : {true_label}")
print("Predictions:")
for r in results[:3]:
    print(f"  {r['label']}: {r['score']:.4f}")

---
## Model card summary (copy to README)

Run this cell to get a ready-made table for the README comparison section.

In [ ]:
from transformers import pipeline as hf_pipeline
from transformers import CLIPProcessor, CLIPModel
import openai, base64, json, os
from io import BytesIO

LABELS_DISPLAY = list(DISPLAY_NAMES.values())
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"

vit_clf = hf_pipeline("image-classification", model=HF_REPO_ID)
clip_proc = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).eval()

def clip_predict(img):
    texts = [f"a photo of a {l} in an apartment" for l in LABELS_DISPLAY]
    inputs = clip_proc(text=texts, images=img, return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = clip_model(**inputs).logits_per_image[0].softmax(dim=0)
    top3 = probs.topk(3)
    return [(LABELS_DISPLAY[i], float(probs[i])) for i in top3.indices]

def openai_predict(img):
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return [("API key missing", 0.0)]
    client = openai.OpenAI(api_key=api_key)
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    labels_str = ", ".join(f'"{l}"' for l in LABELS_DISPLAY)
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": [
            {"type": "text", "text": f'Classify this room. Labels: [{labels_str}]. JSON only: {{"label": "...", "confidence": 0.0}}'},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}],
        max_tokens=80, temperature=0,
    )
    r = json.loads(resp.choices[0].message.content.strip().strip("`").lstrip("json").strip())
    return [(r["label"], r["confidence"])]

print("| Image | True Class | ViT Top-3 | CLIP Top-3 | OpenAI |")
print("|---|---|---|---|---|")

for i in range(5):
    sample = dataset["test"][i]
    img = sample["image"].convert("RGB")
    true = DISPLAY_NAMES.get(id2label[sample["label"]], id2label[sample["label"]])

    vit_top3 = vit_clf(img)[:3]
    vit_str = "<br>".join(f"`{r['label']}` ({r['score']:.4f})" for r in vit_top3)

    clip_top3 = clip_predict(img)
    clip_str = "<br>".join(f"`{l}` ({s:.4f})" for l, s in clip_top3)

    oai = openai_predict(img)
    oai_str = f"`{oai[0][0]}` ({oai[0][1]:.2f})"

    print(f"| img_{i} | `{true}` | {vit_str} | {clip_str} | {oai_str} |")